In [1]:
## conda env!!!

In [2]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'acme'
ORGANISMS_TO_USE = ['Human']
CLASSES_TO_USE = ['I']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01', 'HLA_B_27_05', 'HLA_A_03_01', 'HLA_B_07_02']

In [3]:
# importing the predict module
from mhcnames import normalize_allele_name## ONLY FOR LINUX

import pandas as pd
import os
import sys

In [4]:
# Read datasets

DATA_PATH = "/home/denklewer/Data/"
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/'+ METHOD+'/'

DECOY_KEY = 'x1'
# INPUT_DATA_PATH = DATA_PATH + 'input/'
# per_allele_val_df = {
#     allele_name :pd.read_csv("{}df_{}_{}_val_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
# }



In [5]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)
per_allele_df = per_allele_val_df

/home/denklewer/Tools/anacoda3/envs/acmemhc/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2818: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


In [6]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [6]:
for allele_name in ALLELES:
    per_allele_val_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_val_df[allele_name].index.astype(str).values
#     per_allele_val_df[allele_name]['length'] = per_allele_val_df[allele_name]['peptide'].str.len()
    
#     #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
#     if defineClass(allele_name) == 'I':
#         per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 15]
#     else:
#         per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 30]

In [7]:
per_allele_val_df[ALLELES[0]].head()

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,gene_symbol,n_samples,n_studies,Sample_ids,sample_names,Study_ids,study_names,negative_ensembl_id,binder,source_sequence_name
52052,test,1,HLA-A*02:01,AGIALNDNFVK,DTHSSIFDAK,LISWYDNEYG,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,GAPDHS,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1,seq52052
52053,test,1,HLA-A*02:01,AGLYRGLCAVRSR,GGAGPAVLQV,ALGLGLVSPA,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ATPAF1,1.0,1.0,1320,iedb_1029626_EL_A0201,5,iedb_consolidated,NaN,1,seq52053
52054,test,1,HLA-A*02:01,AIDEKVQAV,APDIDKLKMA,HNLKMFWQST,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,TTC28,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1,seq52054
52055,test,1,HLA-A*02:01,AIFSAIIFL,VRHCFFTVIF,SFEAVSSLSK,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,RHBDD2,1.0,1.0,677,Sarkizova_A0201,21,sarkizova_mhcI_2020,NaN,1,seq52055
52056,test,1,HLA-A*02:01,AIIGGTFTV,PLYRFITTIC,AGILDSCIFT,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ERGIC1,2.0,1.0,"1356,1887","iedb_1019570_EL_A0201,iedb_1034377_EL_A0201",5,iedb_consolidated,NaN,1,seq52056


In [8]:
[normalize_allele_name(allele_name).replace('HLA-', '') for allele_name in ALLELES]

['A*02:01', 'B*27:05', 'A*03:01', 'B*07:02']

## Make Df for training
species, mhc, length, sequence, inequality, affinity (we have no affinity so <500 for binders and >450000 for non-binders
make 1 df at the end

In [18]:
for allele_name in ALLELES:
    df =  per_allele_train_df[allele_name]
    df['species'] = 'human'
    df['mhc'] =  df['allele'].values
    df['length'] = df['peptide'].str.len().values
    df['sequence']  =  df['peptide'].values
    df.loc[df['binder'] == 0,'inequality'] = '>'
    df.loc[df['binder'] == 0,'meas'] = 50000
    df.loc[df['binder'] == 1,'inequality'] = '<'
    df.loc[df['binder'] == 1,'meas'] = 500
    per_allele_train_df[allele_name] = df

train_total_df = pd.concat(per_allele_train_df.values())
train_total_df.head()

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,Study_ids,study_names,negative_ensembl_id,binder,species,mhc,length,inequality,meas,sequence
21410,train,1,HLA-A*03:01,AAAKPTPPK,PKPIFFNLNI,SQVTLTKEFP,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,NaN,NaN,...,5,iedb_consolidated,NaN,1,human,HLA-A*03:01,9,<,500.0,AAAKPTPPK
21411,train,1,HLA-A*03:01,AAAKVILKK,CRVINKEHVL,SRG,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,NaN,NaN,...,5,iedb_consolidated,NaN,1,human,HLA-A*03:01,9,<,500.0,AAAKVILKK
21412,train,1,HLA-A*03:01,AAFVPLLLK,MELLDGKQTL,VCNNPGLYSN,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,NaN,NaN,...,21,sarkizova_mhcI_2020,NaN,1,human,HLA-A*03:01,9,<,500.0,AAFVPLLLK
21413,train,1,HLA-A*03:01,AANKVVLQLK,ELQNLIQSLE,$,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,NaN,NaN,...,5,iedb_consolidated,NaN,1,human,HLA-A*03:01,10,<,500.0,AANKVVLQLK
21414,train,1,HLA-A*03:01,AAREQQILEK,ERREEREKCL,QKRARLQYEK,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,YFAMYQENVAQTDVDTLYIIYRDYTWAELAYTWY,NaN,NaN,...,5,iedb_consolidated,NaN,1,human,HLA-A*03:01,10,<,500.0,AAREQQILEK


In [23]:
len(train_total_df)

52052

In [ ]:
PREDICTOR_HOME = os.getenv('ACME_HOME')
sys.path.append(PREDICTOR_HOME)
import acme_config # method has hardcoded paths in config

In [22]:
PREDICTOR_HOME = os.getenv('ACME_HOME')

## create tempfiles for acme
PATH_TO_BINDING_DATA = "{}/{}".format(PREDICTOR_HOME, acme_config.path_train)
train_total_df[['species', 'mhc', 'length', 'sequence', 'inequality','meas' ]].to_csv(PATH_TO_BINDING_DATA, sep='\t',index=False)


## run main.py of acme   (need more memory)